In [1]:
!pip install bert-score transformers datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import time
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from bert_score import score as bert_score
import re

In [ ]:
from huggingface_hub import login
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# OPCIONAL: autentícate si el modelo está restringido
login(token="")  # Descomenta esta línea y reemplaza con tu token

# Cargar modelo base
base_model = AutoModelForCausalLM.from_pretrained(
    "NousResearch/Nous-Hermes-llama-2-7b",
    device_map="auto",           # Para usar múltiples GPUs si están disponibles
    trust_remote_code=True       # Necesario si el modelo usa código personalizado
)

# Cargar el modelo adaptado con PEFT (LoRA u otro método)
model = PeftModel.from_pretrained(
    base_model,
    "raulgdp/Llama-2-7B-Nous-Hermes-llama-JEP",
    device_map="auto"
)

# (Opcional) cargar el tokenizer si lo necesitas para inferencia
tokenizer = AutoTokenizer.from_pretrained("NousResearch/Nous-Hermes-llama-2-7b")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/13.5G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/175 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuratio

...

tokenizer_config.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

In [4]:
# Dataset de prueba desde HuggingFace
dataset = load_dataset("jdavit/colombian-conflict-SQA", split="test")



README.md:   0%|          | 0.00/485 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/19.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2895 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/30 [00:00<?, ? examples/s]

In [5]:
# Generar respuestas con el modelo y medir latencia
records = []

for item in dataset:
    question = item["question"]
    context = item["context"]
    answer = item["answer"]

    prompt = f"""A continuación, se presenta una pregunta sobre el conflicto armado colombiano, junto con un contexto que proporciona información relevante. Escribe una respuesta que complete adecuadamente la solicitud.

Pregunta:
{question}

Contexto:
{context}

Respuesta:
"""

    start_time = time.time()
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=150)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True).split("Respuesta:")[-1].strip()
    latency = time.time() - start_time
    print("Respuesta generada con exito: ", response)

    records.append({
        "question": question,
        "context": context,
        "expected_answer": answer,
        "predicted_answer": response,
        "latency": round(latency, 3)
    })

df = pd.DataFrame(records)


Respuesta generada con exito:  El informe documenta que la Operación Orión fue un operativo militar que incluyó la participación de fuerzas de seguridad colombianas y paramilitares, con el objetivo de desarticular a las organizaciones armadas ilegales y restablecer el orden en la Comuna 13 de Medellín. El informe señala que durante la operación se reportaron violaciones a los derechos humanos, incluyendo desapariciones forzadas, detenciones arbitrarias y desplazamientos forzados. Además, se resalta la resistencia de las comunidades afectadas, quienes organizaron protestas y resistencias para denunciar
Respuesta generada con exito:  El informe destaca que la expansión de los monocultivos de palma aceitera y caña de azúcar en Colombia ha sido un factor clave en la profundización del conflicto armado, generando desplazamientos forzados y pérdida de tierras por parte de comunidades indígenas y campesinas. Además, señala que la falta de protección estatal y la presencia de grupos armados il

In [6]:
# Normalización básica para F1
def normalize_text(text):
    return re.sub(r'\W+', ' ', text.strip().lower())

def compute_f1(pred, gold):
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()
    common = set(pred_tokens) & set(gold_tokens)
    if len(common) == 0:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)
    return 2 * (precision * recall) / (precision + recall)

# Calcular métricas por fila
bert_precisions, bert_recalls, bert_f1s = bert_score(
    df["predicted_answer"].tolist(),
    df["expected_answer"].tolist(),
    lang="es",  # Asumiendo español
    verbose=True
)

df["bert_score"] = [round(f.item(), 3) for f in bert_f1s]
df["f1_score"] = [round(compute_f1(pred, gold), 3)
                  for pred, gold in zip(df["predicted_answer"], df["expected_answer"])]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.40 seconds, 74.63 sentences/sec


In [7]:
df.to_excel("Resultados-Llama-2-7B-Nous-Hermes-llama-JEP.xlsx")